# 作品① · 抖音数据首次分析

目标：用 pandas 读自己的 17 条作品数据，回答一个问题——**什么样的内容表现更好？**

用法：从上到下，点每个代码格的 ▶ 运行。先看输出，再看下面代码在干嘛。

## 第 1 步：读数据，看前 5 行

`read_excel` 读表格，`head()` 看前 5 行。运行后看看每列是什么。

In [ ]:
import pandas as pd

df = pd.read_excel('C:/Users/Administrator/Desktop/作品列表导出.xlsx')
print('数据大小：', df.shape)
df.head()

## 第 2 步：认识数据

`shape` 是（行数, 列数）。17 行作品，19 列字段。

运行 `df.info()` 看每一列的类型，有没有空值。

In [ ]:
print('行数', df.shape[0], '列数', df.shape[1])
df.info()

## 第 3 步：清洗数据（很重要）

两个坑必须先处理，否则结论会错：

1. **6 条「自见」作品播放量是 0**，是自己可见的，要排除掉，只留「公开」。
2. `cover_ctr` 列有个 `-` 是无效值，`to_numeric(errors='coerce')` 把它变成空值，不影响别的计算。

In [ ]:
print('清洗前：')
print(df['audit_status'].value_counts())

df = df[df['audit_status'] == '公开'].copy()
df['cover_ctr'] = pd.to_numeric(df['cover_ctr'], errors='coerce')

print('清洗后剩', len(df), '条公开作品')

## 第 4 步：播放量最高的 5 条

`sort_values` 排序，`ascending=False` 是从高到低。

看看排第一的那条播放量是多少——它比第二名高了多少倍？

In [ ]:
top5 = df.sort_values('views', ascending=False)
top5[['title', 'views', 'finish_rate', 'likes']].head()

## 第 5 步：一个必须知道的坑——平均数会被爆款拉高

那条 298 万播放的爆款，会把「平均播放量」拉得特别高，用平均数判断自己通常的水平会失真。

**教程反复强调：用率不用绝对数，中位数比平均数可靠。** 运行看看两者差多少。

In [ ]:
print('平均播放量：', round(df['views'].mean()))
print('中位数播放量：', round(df['views'].median()))

## 第 6 步：算互动率，找真正的好内容

互动率 =（点赞+分享+评论+收藏）÷ 播放量。

它比绝对点赞数公平——播放量小的作品，互动率可能反而更高。

In [ ]:
df['互动率'] = (df['likes'] + df['shares'] + df['comments'] + df['favs']) / df['views']

df[['title', 'views', '互动率', 'finish_rate']].sort_values('互动率', ascending=False).head()

## 第 7 步：哪条最涨粉

播放量高不等于涨粉多。看看涨粉排名和播放量排名是不是同一批作品。

In [ ]:
df[['title', 'views', 'new_followers']].sort_values('new_followers', ascending=False).head()

## 第 8 步：写下你的观察（这是你的成果）

在下面用文字写 2-3 句，比如：

- 播放量最高的是哪条？它和其他条差多少？
- 互动率高的作品，是爆款还是小播放量的？
- 涨粉最多的那条，选题是什么？（等你填了 topic 列再回来补充）

**把这几点写进「作品①计划书」的第三部分，计划书就活了。**

## 我的观察（2026-11-16，第一次分析）

**1. 数据概况**
17 条作品里有 6 条是「自见」（自己可见、播放量为 0），排除后真正能分析的是 **11 条公开作品**。

**2. 存在一个超级爆款，必须用中位数看水平**
「#大学生」那条播放量 **2,981,390**（约 298 万），是第二名「#聊天记录」（10,184）的 **293 倍**。
- 平均播放量：**274,035**
- 中位数播放量：**1,112**

两者差 246 倍——平均数被那一条爆款完全拉飞了。所以我平时的真实水平要看**中位数 1,112**，而不是 27 万这个虚高的平均数。这就是教程说的「用率不用绝对数、警惕爆款」的活例子。

**3. 互动率（赞+评+转+藏 ÷ 播放）**
- 「#大学生」：10.27%
- 「#自我认知 #文字语录」：8.41%（播放量只有 642，但互动率排第二）
- 「#个人成长记录」：5.13%（播放量 156）

小播放量的作品互动率反而很高，说明「愿不愿意互动」和「有多少人看到」是两回事。不过这几条播放量太小，样本不足，先记下、不下定论。

**4. 涨粉几乎全靠那一条爆款**
「#大学生」一条带来 **251 个新粉丝**，其余作品涨粉最多只有 1 个。说明我的日常内容涨粉能力很弱，涨粉和爆款强绑定。**这是下一步最值得改的问题**：怎么让非爆款内容也能涨粉。

**5. 计划书的三个假设，目前验证了几个？**
- 假设①②③都**还没法验证**——因为 topic（选题）、hook（钩子）两列还是空的，无法按选题/钩子分组。
- **下一步**：先给 11 条公开作品手工补上 topic/hook，再回来跑一次分组，就能回答「哪类选题播放高、哪种钩子留人」了。

**一句话结论**：现在最该做的不是追播放量，而是①补 topic/hook 让分析能分组，②研究那条 298 万爆款为什么涨粉 251、把它的钩子复用到其他作品。
